# DINOv2 Backbone Verification

**Model**: `facebook/dinov2-{small|base|large|giant}` (Meta AI)  
**Arch**: ViT-S/14 (22M) · ViT-B/14 (86M) · ViT-L/14 (307M) · ViT-G/14 (1.1B)  
**Embed dim**: 384 · 768 · 1024 · 1536  
**Input**: 3×224×224 px, ImageNet normalization (via `AutoImageProcessor`)  
**Access**: HuggingFace, no auth  
**Zero-shot**: No — vision-only self-supervised pretraining

Paper: [DINOv2: Learning Robust Visual Features without Supervision (arXiv 2304.07193)](https://arxiv.org/abs/2304.07193)

In [ ]:
%pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

# Pick a variant
VARIANT = "large"   # small | base | large | giant
HUB = f"facebook/dinov2-{VARIANT}"

EMBED_DIMS = {"small": 384, "base": 768, "large": 1024, "giant": 1536}

processor = AutoImageProcessor.from_pretrained(HUB)
model     = AutoModel.from_pretrained(HUB)
model.eval()

print("Hub:", HUB)
print("Expected embed dim:", EMBED_DIMS[VARIANT])
print("Patch size:", model.config.patch_size)        # 14
print("Image size:", model.config.image_size)        # 518 for large/giant, 224 for s/b

## 2. Preprocess image

In [ ]:
from PIL import Image
import numpy as np

# AutoImageProcessor handles resize + center-crop + normalize automatically
# CXR-specific: load as grayscale then convert to RGB (3-channel repeat)
def preprocess(path: str) -> dict:
    img = Image.open(path).convert("RGB")   # grayscale → 3-ch via PIL RGB
    return processor(images=img, return_tensors="pt")

# Show processor settings
print("Crop size:  ", processor.crop_size)
print("Image mean: ", processor.image_mean)
print("Image std:  ", processor.image_std)

# Dummy smoke-test
dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
inputs = processor(images=dummy_pil, return_tensors="pt")
print("pixel_values:", inputs["pixel_values"].shape)

## 3. Extract embeddings

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

cls_emb   = outputs.last_hidden_state[:, 0]    # [B, D] — CLS token
pooled    = outputs.pooler_output               # [B, D] — same as CLS (layernorm applied)
patches   = outputs.last_hidden_state[:, 1:]   # [B, N_patches, D]
mean_emb  = patches.mean(dim=1)                # [B, D] — mean-pool patches

print("CLS / pooled:  ", cls_emb.shape)    # [1, 1024]
print("patch tokens:  ", patches.shape)    # [1, 1370, 1024] for 518px ViT-L/14
print("mean pool:     ", mean_emb.shape)

## 4. Multi-scale features with `get_intermediate_layers`

In [ ]:
# Useful for dense tasks (segmentation, detection)
# n=4 returns last 4 transformer block outputs
with torch.no_grad():
    feats = model.get_intermediate_layers(
        inputs["pixel_values"], n=4, return_class_token=True
    )  # list of (patch_tokens, cls_token) tuples

for i, (patch, cls) in enumerate(feats):
    print(f"  layer -{4-i}: patches={patch.shape}  cls={cls.shape}")

## Segmentation mode — spatial feature map

Reshape the patch sequence (section 3) to a 2-D spatial grid for dense prediction.
- **ViT-L/14 at 518×518**: `[B, 1024, 37, 37]` (518/14 ≈ 37, 37²=1369 patches)
- **ViT-B/14 at 518×518**: `[B, 768, 37, 37]`
- **ViT-B/14 at 224×224**: `[B, 768, 16, 16]` (224/14 ≈ 16, 16²=256 patches)

In [ ]:
import math

# patches from section 3: outputs.last_hidden_state[:, 1:]
# Reshape to [B, D, H, W] spatial feature map for dense prediction heads.
B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 37 for 518px / 14px-patches (ViT-L); 16 for 224px (ViT-B)
assert H * W == N, f"Patch grid {N} is not a perfect square"

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 1369, 1024] for ViT-L at 518px
print(f"spatial map:   {spatial.shape}")   # [1, 1024, 37, 37]

## 5. Compare all variants

In [ ]:
for variant in ["small", "base", "large"]:   # skip giant unless you have 80GB
    hub = f"facebook/dinov2-{variant}"
    proc = AutoImageProcessor.from_pretrained(hub)
    mdl  = AutoModel.from_pretrained(hub).eval()
    inp  = proc(images=dummy_pil, return_tensors="pt")
    with torch.no_grad():
        out = mdl(**inp)
    print(f"dinov2-{variant}: pooled={out.pooler_output.shape}  "
          f"n_patches={out.last_hidden_state.shape[1]-1}  "
          f"input={inp['pixel_values'].shape[-2:]}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).pooler_output   # CLS [B, D]
pool       : None
embed_dim  : 1024 (large) / 768 (base) / 384 (small) / 1536 (giant)
transform  : AutoImageProcessor — handles resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

DINOv2 is already used as the backbone for RAD-DINO (`microsoft/rad-dino`),
which is a DINOv2 ViT-B/14 fine-tuned on CXR. Running vanilla DINOv2 (not
CXR-adapted) here provides a **domain gap baseline** — useful for ablation.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model(pixel_values=x).last_hidden_state[:, 1:]` → `[B, N, D]`
- Spatial map: reshape to `[B, D, 37, 37]` for ViT-L/14 at 518px, `[B, D, 16, 16]` for ViT-B/14 at 224px
- `get_intermediate_layers(x, n=4)` for multi-scale features (section 4)

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

inputs = preprocess(CXR_PATH)
with torch.no_grad():
    out = model(**inputs)

emb = out.pooler_output   # [1, D]
print("embedding:", emb.shape, "  norm:", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).pooler_output   # CLS [B, D]
pool       : None
embed_dim  : 1024 (large) / 768 (base) / 384 (small) / 1536 (giant)
transform  : AutoImageProcessor — handles resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

DINOv2 is already used as the backbone for RAD-DINO (`microsoft/rad-dino`),
which is a DINOv2 ViT-B/14 fine-tuned on CXR. Running vanilla DINOv2 (not
CXR-adapted) here provides a **domain gap baseline** — useful for ablation.